# 03 — Full Experiment

Runs all 5 queries from EXPERIMENT.md, computes nDCG@3 and latency for each
retrieval system (BM25, Vector, Hybrid), and produces a comparison chart.

**Prerequisites:**
1. `01_data_collection.ipynb` — generates `../data/chunks.json`
2. `02_retrieval_engines.ipynb` — confirms all engines work correctly

**Output:** `../data/comparison_chart.png` — grouped bar chart of nDCG@3 scores.

**Reference:** EXPERIMENT.md §Experiment Queries and §Evaluation Method

In [ ]:
# Cell 1 — Install dependencies and imports
!pip install rank-bm25 sentence-transformers faiss-cpu matplotlib pandas

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.chunker import load_chunks
from src.bm25_retriever import BM25Retriever
from src.vector_retriever import VectorRetriever
from src.hybrid_retriever import HybridRetriever
from src.evaluate import compute_ndcg, measure_latency, plot_comparison, print_results_table

In [ ]:
# Cell 2 — Load data and build indexes
chunks = load_chunks("../data/chunks.json")

bm25 = BM25Retriever()
bm25.build_index(chunks)

vector = VectorRetriever()
vector.build_index(chunks)

hybrid = HybridRetriever(bm25, vector)

In [ ]:
# Cell 3 — Define experiment queries
# Five queries designed to stress-test different retrieval capabilities
# (EXPERIMENT.md §Experiment Queries)
QUERIES = {
    "Q1": "How many goals did Messi score in the 2022 World Cup?",
    "Q2": "Which player is known for creative dribbling and exceptional vision?",
    "Q3": "What country hosted the 2026 FIFA World Cup?",
    "Q4": "Tell me about a goalkeeper famous for saving penalty kicks",
    "Q5": "Who is the youngest player with the most World Cup appearances?"
}

print("Experiment queries:")
for label, query in QUERIES.items():
    print(f"  {label}: {query}")

In [ ]:
# Cell 4 — Run all retrievers, print Top-3 per query
TOP_K = 3

# Store results for later nDCG computation
all_results = {}

for label, query in QUERIES.items():
    bm25_res   = bm25.search(query, top_k=TOP_K)
    vector_res = vector.search(query, top_k=TOP_K)
    hybrid_res = hybrid.search(query, top_k=TOP_K)
    all_results[label] = {'bm25': bm25_res, 'vector': vector_res, 'hybrid': hybrid_res}

    print(f"\n{'='*70}")
    print(f"{label}: {query}")
    print(f"{'='*70}")

    # Print side-by-side table
    col_w = 38
    header = f"{'Rank':<5}| {'BM25':<{col_w}}| {'Vector':<{col_w}}| {'Hybrid'}"
    print(header)
    print("-" * (5 + 1 + col_w + 1 + col_w + 1 + col_w + 1))

    for i in range(TOP_K):
        def fmt(results, idx, col_w=col_w):
            if idx >= len(results): return "(none)"
            r = results[idx]
            return f"[{r['source'][:12]}] {r['text'][:18]}..."[:col_w]

        row = f"{i+1:<5}| {fmt(bm25_res, i):<{col_w}}| {fmt(vector_res, i):<{col_w}}| {fmt(hybrid_res, i)}"
        print(row)

    print()
    print("Detailed Top-1 results:")
    print(f"  BM25   chunk_id={bm25_res[0]['chunk_id']} | {bm25_res[0]['source']}")
    print(f"    {bm25_res[0]['text'][:120]} ...")
    print(f"  Vector chunk_id={vector_res[0]['chunk_id']} | {vector_res[0]['source']}")
    print(f"    {vector_res[0]['text'][:120]} ...")
    print(f"  Hybrid chunk_id={hybrid_res[0]['chunk_id']} | {hybrid_res[0]['source']}")
    print(f"    {hybrid_res[0]['text'][:120]} ...")

In [ ]:
# Cell 5 — Manual relevance annotation
#
# Fill in after inspecting Cell 4 results.
# Scoring: 2 = directly answers the query, 1 = related, 0 = irrelevant
# Keys are chunk_id integers from the results above.
#
# NOTE: This cell requires human judgment.
# Replace each TODO with the actual chunk_id → score mapping.

RELEVANCE = {
    "Q1": {},  # TODO: fill after running Cell 4
                # e.g. {42: 2, 17: 1, 89: 0}
    "Q2": {},  # TODO: fill after running Cell 4
    "Q3": {},  # TODO: fill after running Cell 4
    "Q4": {},  # TODO: fill after running Cell 4
    "Q5": {},  # TODO: fill after running Cell 4
}

print("RELEVANCE template created — fill in chunk_id -> score mappings above.")
print("Then re-run this cell and proceed to Cell 6 for nDCG computation.")

In [ ]:
# Cell 6 — Compute nDCG@3 for all systems
ndcg_results = {}

for label in QUERIES:
    rel = RELEVANCE[label]
    ndcg_results[label] = {
        'bm25':   compute_ndcg(all_results[label]['bm25'],   rel, k=3),
        'vector': compute_ndcg(all_results[label]['vector'], rel, k=3),
        'hybrid': compute_ndcg(all_results[label]['hybrid'], rel, k=3),
    }

print("nDCG@3 Results:")
print_results_table(ndcg_results)

In [ ]:
# Cell 7 — Latency measurement
query_list = list(QUERIES.values())

latency_bm25   = measure_latency(bm25,   query_list, top_k=5, runs=10)
latency_vector = measure_latency(vector, query_list, top_k=5, runs=10)
latency_hybrid = measure_latency(hybrid, query_list, top_k=5, runs=10)

print(f"{'System':<10} | {'Avg Latency (ms)':>17}")
print("-" * 30)
print(f"{'BM25':<10} | {latency_bm25:>17.2f}")
print(f"{'Vector':<10} | {latency_vector:>17.2f}")
print(f"{'Hybrid':<10} | {latency_hybrid:>17.2f}")

In [ ]:
# Cell 8 — Visualisation
# Saves grouped bar chart comparing BM25, Vector, Hybrid across Q1–Q5.
# Chart is saved to ../data/comparison_chart.png
plot_comparison(ndcg_results, save_path="../data/comparison_chart.png")

# Cell 9 — Summary and Conclusions

## Results Summary

*(Fill in after running Cells 4–8 with actual nDCG scores)*

### Keyword Queries (Q1, Q3)

**Q1** — *"How many goals did Messi score in the 2022 World Cup?"*  
Expected winner: **BM25** — the exact terms "Messi", "goals", "2022 World Cup" appear verbatim in Wikipedia.
- BM25 should retrieve the relevant chunk with high precision.
- Vector search may return semantically similar but numerically imprecise chunks.
- Hybrid preserves BM25 advantage while reducing noise.

**Q3** — *"What country hosted the 2026 FIFA World Cup?"*  
Expected winner: **BM25** — exact noun phrase "2026 FIFA World Cup" and "host" appear in structured Wikipedia text.

### Semantic Queries (Q2, Q4)

**Q2** — *"Which player is known for creative dribbling and exceptional vision?"*  
Expected winner: **Vector** — the phrase "creative dribbling" may not appear verbatim in any chunk, but the
embedding space maps it close to Messi/Neymar player descriptions.

**Q4** — *"Tell me about a goalkeeper famous for saving penalty kicks"*  
Expected winner: **Vector** — semantic similarity between "penalty saving" and goalkeeper descriptions
drives retrieval without requiring exact phrase matches.

### Mixed Query (Q5)

**Q5** — *"Who is the youngest player with the most World Cup appearances?"*  
Expected winner: **Hybrid** — requires both exact stat lookup (BM25 strength) and contextual
understanding of "youngest" in a historical context (Vector strength).  
Neither alone is sufficient; RRF fusion captures both signals.

---

## Connection to B+ Tree Indexing

This experiment validates the theoretical argument from CONCEPT.md:

| Era | Index | Search Key | Demonstrated By |
|-----|-------|------------|-----------------|
| Classic | B+ Tree | Ordered scalar (goals: int) | Exact stat queries (Q1, Q3) |
| Modern text | BM25 / Inverted Index | Term (word) | Q1, Q3 BM25 strength |
| AI era | Vector Index (FAISS) | Embedding vector | Q2, Q4 Vector strength |
| 2024–present | **Hybrid RAG** | Both | Q5 Hybrid dominance |

> **Key insight:** The B+ Tree requires a *total order* on keys — it can answer
> "find goals > 5" but cannot answer "find a creative dribbler."  Vector indexes
> replace the order relation with proximity in semantic space.  Hybrid RAG combines
> both, matching the intuition that a complete retrieval system should handle
> **all** query types — not just the ones its index structure was designed for.

## Hypothesis Evaluation (from EXPERIMENT.md)

| Hypothesis | Status |
|-----------|--------|
| H1: BM25 ≥ Vector on Q1, Q3 | *(check after running)* |
| H2: Vector ≥ BM25 on Q2, Q4 | *(check after running)* |
| H3: Hybrid ≥ max(BM25, Vector) on all queries | *(check after running)* |
| H4: Hybrid strictly > both on Q5 | *(check after running)* |